## Pool results from CLIF consortium sites (24h + 72h windows)

In [ ]:
# =============================================================
# CONFIG
# =============================================================
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

INPUT_DIR = Path(r'C:\Users\ntippan\Downloads\AHA-OHCA\AHA-OHCA')
OUTPUT_DIR = Path(r'C:\Users\ntippan\ALL_CODES\Git_folders\AHA-grant\OHCA-vitals_trajectory\pooled_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOWS = [24, 72]
VITALS = ['heart_rate', 'temp_c', 'map', 'spo2']
TRAJ_ORDER = ['Group 1', 'Group 2', 'Group 3', 'Group 4']

TRAJ_COLORS = {
    'Group 1': '#1565C0',
    'Group 2': '#4CAF50',
    'Group 3': '#FF9800',
    'Group 4': '#E53935',
}
SURV_COLORS = {'Survivor': '#2196F3', 'Non-Survivor': '#E53935'}

# List all site folders
site_dirs = sorted([d for d in INPUT_DIR.iterdir() if d.is_dir() and d.name != 'pooled_results'])
print(f'Found {len(site_dirs)} sites:')
for d in site_dirs:
    print(f'  {d.name}')

In [ ]:
def load_site_files(input_dir, file_pattern, window, site_label_map=None):
    """Load a specific file from each site's Upload_to_Box folder.
    
    If site_label_map is provided, remap site names to anonymous labels.
    """
    input_dir = Path(input_dir)
    subfolder = f'Upload_to_Box_without_oral_{window}'
    frames = []
    missing = []
    
    for site_dir in sorted(input_dir.iterdir()):
        if not site_dir.is_dir() or site_dir.name == 'pooled_results':
            continue
        fpath = site_dir / subfolder / file_pattern
        if fpath.exists():
            df = pd.read_csv(fpath)
            label = site_label_map.get(site_dir.name, site_dir.name) if site_label_map else site_dir.name
            if 'site' not in df.columns:
                df.insert(0, 'site', label)
            else:
                df['site'] = label
            frames.append(df)
            print(f'  [OK] {label:15s}: {len(df):>6,} rows')
        else:
            missing.append(site_dir.name)
            print(f'  [--] {site_dir.name:15s}: NOT FOUND')
    
    if frames:
        combined = pd.concat(frames, ignore_index=True)
        print(f'\n  Total: {len(combined):,} rows from {len(frames)} sites')
        if missing:
            print(f'  Missing: {missing}')
        return combined
    return pd.DataFrame()


def build_site_label_map(input_dir, window):
    """
    Scan each site folder, read table1_poolable to get cohort N,
    then assign A, B, C, ... by descending cohort size.
    """
    import string
    input_dir = Path(input_dir)
    subfolder = f'Upload_to_Box_without_oral_{window}'
    site_sizes = {}
    
    for site_dir in sorted(input_dir.iterdir()):
        if not site_dir.is_dir() or site_dir.name == 'pooled_results':
            continue
        fpath = site_dir / subfolder / f'table1_poolable_{window}h.csv'
        if fpath.exists():
            df = pd.read_csv(fpath)
            n_row = df[(df['group'] == 'Overall') & (df['variable'] == 'n')]
            n = int(n_row['value'].iloc[0]) if len(n_row) > 0 else 0
            site_sizes[site_dir.name] = n
    
    # Rank by descending cohort size → A = largest
    ranked = sorted(site_sizes.items(), key=lambda x: x[1], reverse=True)
    label_map = {}
    for i, (site_name, n) in enumerate(ranked):
        label = string.ascii_uppercase[i] if i < 26 else f'Site{i+1}'
        label_map[site_name] = label
        print(f'  {site_name:20s} → Site {label}  (n={n:,})')
    
    return label_map
    
TRAJ_REMAP = {
    'Hypothermic':    'Group 1',
    'Normothermic':   'Group 2',
    'Rapid Decline':  'Group 3',
    'Persistent High':'Group 4',
}

data = {}
for window in WINDOWS:
    print(f'\n{"=" * 60}')
    print(f'  LOADING {window}h WINDOW')
    print(f'{"=" * 60}')
    
    print(f'\n  --- Site label assignment (by cohort size) ---')
    site_label_map = build_site_label_map(INPUT_DIR, window)
    
    print(f'\n  --- table1_poolable_{window}h.csv ---')
    data[f't1_{window}'] = load_site_files(INPUT_DIR, f'table1_poolable_{window}h.csv', window, site_label_map)
    
    print(f'\n  --- hourly_vitals_by_trajectory_survival_{window}h.csv ---')
    data[f'hv_{window}'] = load_site_files(INPUT_DIR, f'hourly_vitals_by_trajectory_survival_{window}h.csv', window, site_label_map)
    
    # Remap old trajectory names → Group 1–4 in both dataframes
    for key in [f't1_{window}', f'hv_{window}']:
        df = data[key]
        if len(df) == 0:
            continue
        if 'group' in df.columns:
            df['group'] = df['group'].replace(TRAJ_REMAP)
        if 'trajectory' in df.columns:
            df['trajectory'] = df['trajectory'].replace(TRAJ_REMAP)

# NOW check counts
for window in WINDOWS:
    t1 = data[f't1_{window}']
    hv = data[f'hv_{window}']
    
    print(f'\n{"=" * 60}')
    print(f'  SITE COUNTS — {window}h')
    print(f'{"=" * 60}')
    
    # --- N, Survivors, Non-Survivors, Mortality per site ---
    print(f'\n  {"Site":15s} {"N":>8s} {"Survivor":>10s} {"Non-Surv":>10s} {"Mort%":>8s} {"N(Vitals)":>10s}')
    print(f'  {"-"*15} {"-"*8} {"-"*10} {"-"*10} {"-"*8} {"-"*10}')
    
    hv_n = hv[hv['hour'] == 0].groupby('site')['n_heart_rate'].sum().reset_index()
    hv_n.columns = ['site', 'n_vitals']
    
    totals = {'n': 0, 'surv': 0, 'nonsurv': 0, 'n_vitals': 0}
    
    for site in sorted(t1['site'].unique()):
        sd = t1[(t1['site'] == site)]
        
        def gv(group, var):
            match = sd[(sd['group'] == group) & (sd['variable'] == var)]['value']
            return int(match.iloc[0]) if len(match) > 0 else 0
        
        n = gv('Overall', 'n')
        surv = gv('Survivor', 'n')
        nonsurv = gv('Non-Survivor', 'n')
        mort_row = sd[(sd['group'] == 'Overall') & (sd['variable'] == 'mortality_pct')]['value']
        mort = float(mort_row.iloc[0]) if len(mort_row) > 0 else 0
        
        nv_row = hv_n[hv_n['site'] == site]['n_vitals']
        nv = int(nv_row.iloc[0]) if len(nv_row) > 0 else 0
        
        print(f'  {site:15s} {n:>8,} {surv:>10,} {nonsurv:>10,} {mort:>7.1f}% {nv:>10,}')
        
        totals['n'] += n
        totals['surv'] += surv
        totals['nonsurv'] += nonsurv
        totals['n_vitals'] += nv
    
    total_mort = totals['nonsurv'] / totals['n'] * 100 if totals['n'] > 0 else 0
    print(f'  {"-"*15} {"-"*8} {"-"*10} {"-"*10} {"-"*8} {"-"*10}')
    print(f'  {"TOTAL":15s} {totals["n"]:>8,} {totals["surv"]:>10,} {totals["nonsurv"]:>10,} '
          f'{total_mort:>7.1f}% {totals["n_vitals"]:>10,}')
    
    # Sanity check
    if totals['surv'] + totals['nonsurv'] != totals['n']:
        print(f'\n  ⚠ WARNING: Survivor + Non-Survivor ({totals["surv"] + totals["nonsurv"]:,}) '
              f'!= Overall ({totals["n"]:,})')
    else:
        print(f'\n  ✓ Survivor + Non-Survivor = Overall')

In [ ]:
# =============================================================
# CELL 3.A: QUICK DATA CHECK
# =============================================================
for window in WINDOWS:
    t1 = data[f't1_{window}']
    hv = data[f'hv_{window}']
    
    if len(t1) == 0:
        print(f'\n  {window}h: No table1 data')
        continue
    
    print(f'\n{"=" * 60}')
    print(f'  DATA CHECK: {window}h')
    print(f'{"=" * 60}')
    
    # N per site
    n_per_site = t1[t1['variable'] == 'n'].groupby('site')[['group', 'value']]
    overall_n = t1[(t1['variable'] == 'n') & (t1['group'] == 'Overall')][['site', 'value']].set_index('site')
    print(f'\n  Cohort size per site:')
    for site, row in overall_n.iterrows():
        print(f'    {site:15s}: {int(row["value"]):>6,}')
    print(f'    {"TOTAL":15s}: {int(overall_n["value"].sum()):>6,}')
    
    # Trajectory distribution per site
    print(f'\n  Trajectory distribution:')
    print(f'  {"Site":15s}', end='')
    for traj in TRAJ_ORDER:
        print(f' {traj:>15s}', end='')
    print()
    print(f'  {"-"*15}', end='')
    for _ in TRAJ_ORDER:
        print(f' {"-"*15}', end='')
    print()
    
    for site in sorted(t1['site'].unique()):
        site_data = t1[t1['site'] == site]
        print(f'  {site:15s}', end='')
        for traj in TRAJ_ORDER:
            n_val = site_data[(site_data['group'] == traj) & (site_data['variable'] == 'n')]['value']
            n = int(n_val.iloc[0]) if len(n_val) > 0 else 0
            print(f' {n:>15,}', end='')
        print()
    
    # Mortality per site
    print(f'\n  Mortality per site:')
    for site in sorted(t1['site'].unique()):
        site_data = t1[(t1['site'] == site) & (t1['group'] == 'Overall')]
        mort_pct = site_data[site_data['variable'] == 'mortality_pct']['value']
        n_val = site_data[site_data['variable'] == 'n']['value']
        if len(mort_pct) > 0 and len(n_val) > 0:
            print(f'    {site:15s}: {mort_pct.iloc[0]:>5.1f}% (n={int(n_val.iloc[0]):,})')
    
    # Hourly vitals check
    if len(hv) > 0:
        print(f'\n  Hourly vitals: {len(hv):,} rows, {hv["site"].nunique()} sites')
        print(f'  Hours range: {hv["hour"].min():.0f} - {hv["hour"].max():.0f}')
        print(f'  Trajectories: {sorted(hv["trajectory"].unique())}')

In [ ]:
# =============================================================
# CELL 4: SAVE CONCATENATED RAW FILES
# =============================================================
for window in WINDOWS:
    t1 = data[f't1_{window}']
    hv = data[f'hv_{window}']
    
    if len(t1) > 0:
        t1.to_csv(OUTPUT_DIR / f'all_sites_table1_{window}h.csv', index=False)
        print(f'[OK] all_sites_table1_{window}h.csv ({len(t1):,} rows, {t1["site"].nunique()} sites)')
    
    if len(hv) > 0:
        hv.to_csv(OUTPUT_DIR / f'all_sites_hourly_vitals_{window}h.csv', index=False)
        print(f'[OK] all_sites_hourly_vitals_{window}h.csv ({len(hv):,} rows, {hv["site"].nunique()} sites)')

In [ ]:
# =============================================================
# CELL 5: POOL TABLE 1
# =============================================================
def pool_table1(table1_all, window):
    """Pool Table 1 across sites using combined variance formula."""
    groups = table1_all['group'].unique()
    pooled_rows = []

    for group in groups:
        gd = table1_all[table1_all['group'] == group]
        group_type = gd['group_type'].iloc[0]
        base = {'group_type': group_type, 'group': group, 'window_hours': window}

        # --- N: sum ---
        n_rows = gd[gd['variable'] == 'n']
        total_n = n_rows['value'].sum()
        pooled_rows.append({**base, 'variable': 'n', 'value': total_n,
                            'n_sites': len(n_rows), 'sites': ', '.join(sorted(n_rows['site'].unique()))})

        # --- Continuous: pooled mean + SD ---
        for var_prefix in ['age', 'hosp_los', 'icu_los']:
            mean_rows = gd[gd['variable'] == f'{var_prefix}_mean'][['site', 'value']].rename(columns={'value': 'mean'})
            sd_rows = gd[gd['variable'] == f'{var_prefix}_sd'][['site', 'value']].rename(columns={'value': 'sd'})
            n_vals = gd[gd['variable'] == 'n'][['site', 'value']].rename(columns={'value': 'n'})

            if len(mean_rows) == 0:
                continue

            merged = mean_rows.merge(sd_rows, on='site', how='left').merge(n_vals, on='site', how='left')
            merged = merged.dropna(subset=['mean', 'sd', 'n'])
            merged['n'] = merged['n'].astype(float)
            if len(merged) == 0:
                continue

            N = merged['n'].sum()
            pooled_mean = (merged['n'] * merged['mean']).sum() / N
            within = ((merged['n'] - 1) * merged['sd'] ** 2).sum()
            between = (merged['n'] * (merged['mean'] - pooled_mean) ** 2).sum()
            pooled_sd = np.sqrt((within + between) / max(N - 1, 1))

            sites_str = ', '.join(sorted(merged['site'].unique()))
            pooled_rows.append({**base, 'variable': f'{var_prefix}_mean', 'value': round(pooled_mean, 2),
                                'n_sites': len(merged), 'sites': sites_str})
            pooled_rows.append({**base, 'variable': f'{var_prefix}_sd', 'value': round(pooled_sd, 2),
                                'n_sites': len(merged), 'sites': sites_str})

            # Weighted median approximation
            med_rows = gd[gd['variable'] == f'{var_prefix}_median'][['site', 'value']].rename(columns={'value': 'median'})
            if len(med_rows) > 0:
                med_m = med_rows.merge(n_vals, on='site').dropna().sort_values('median')
                if len(med_m) > 0:
                    cumw = med_m['n'].cumsum()
                    approx_med = med_m.loc[cumw >= med_m['n'].sum() / 2, 'median'].iloc[0]
                    pooled_rows.append({**base, 'variable': f'{var_prefix}_median_approx',
                                        'value': round(approx_med, 1), 'n_sites': len(med_m), 'sites': 'weighted'})

            # Q25, Q75 — weighted approximation
            for q, q_label in [(0.25, 'q25'), (0.75, 'q75')]:
                q_rows = gd[gd['variable'] == f'{var_prefix}_{q_label}'][['site', 'value']].rename(columns={'value': 'qval'})
                if len(q_rows) > 0:
                    q_m = q_rows.merge(n_vals, on='site').dropna().sort_values('qval')
                    if len(q_m) > 0:
                        cumw = q_m['n'].cumsum()
                        approx_q = q_m.loc[cumw >= q_m['n'].sum() * q, 'qval'].iloc[0]
                        pooled_rows.append({**base, 'variable': f'{var_prefix}_{q_label}_approx',
                                            'value': round(approx_q, 1), 'n_sites': len(q_m), 'sites': 'weighted'})

        # --- Categorical: sum counts, recompute pct ---
        cat_n_vars = gd[gd['variable'].str.endswith('_n') & ~gd['variable'].isin(['n', 'mortality_n'])]
        for var_name in cat_n_vars['variable'].unique():
            var_rows = gd[gd['variable'] == var_name]
            total_count = var_rows['value'].sum()
            pooled_rows.append({**base, 'variable': var_name, 'value': int(total_count),
                                'n_sites': len(var_rows), 'sites': ', '.join(sorted(var_rows['site'].unique()))})
            pct_var = var_name.replace('_n', '_pct')
            pooled_rows.append({**base, 'variable': pct_var,
                                'value': round(total_count / total_n * 100, 1) if total_n > 0 else 0,
                                'n_sites': len(var_rows), 'sites': 'computed'})

        # --- Mortality: sum ---
        mort_rows = gd[gd['variable'] == 'mortality_n']
        if len(mort_rows) > 0:
            total_mort = mort_rows['value'].sum()
            pooled_rows.append({**base, 'variable': 'mortality_n', 'value': int(total_mort),
                                'n_sites': len(mort_rows), 'sites': ', '.join(sorted(mort_rows['site'].unique()))})
            pooled_rows.append({**base, 'variable': 'mortality_pct',
                                'value': round(total_mort / total_n * 100, 1) if total_n > 0 else 0,
                                'n_sites': len(mort_rows), 'sites': 'computed'})

    return pd.DataFrame(pooled_rows)


# Run pooling
pooled_t1 = {}
for window in WINDOWS:
    t1 = data[f't1_{window}']
    if len(t1) == 0:
        continue
    pooled = pool_table1(t1, window)
    pooled.to_csv(OUTPUT_DIR / f'pooled_table1_{window}h.csv', index=False)
    pooled_t1[window] = pooled
    print(f'[OK] pooled_table1_{window}h.csv ({len(pooled)} rows)')
    
    # Show key results
    print(f'\n  Pooled Overall ({window}h):')
    overall = pooled[pooled['group'] == 'Overall']
    for var in ['n', 'age_mean', 'age_sd', 'mortality_n', 'mortality_pct']:
        row = overall[overall['variable'] == var]
        if len(row) > 0:
            v = row['value'].iloc[0]
            ns = row['n_sites'].iloc[0]
            print(f'    {var:25s}: {v:>10} ({ns} sites)')
    
    print(f'\n  Pooled by Trajectory ({window}h):')
    for traj in TRAJ_ORDER:
        traj_data = pooled[pooled['group'] == traj]
        n_val = traj_data[traj_data['variable'] == 'n']['value']
        mort_val = traj_data[traj_data['variable'] == 'mortality_pct']['value']
        if len(n_val) > 0:
            print(f'    {traj:20s}: n={int(n_val.iloc[0]):>6,}, mort={mort_val.iloc[0]:.1f}%')
    print()

In [ ]:
# =============================================================
# CELL 6: POOL HOURLY VITALS
# =============================================================
def pool_hourly_vitals(hourly_all, window):
    """Pool hourly vitals across sites using combined variance formula."""
    group_cols = ['hour', 'trajectory', 'survival_status']

    # Filter to this window if column exists
    if 'window_hours' in hourly_all.columns:
        hourly_all = hourly_all[hourly_all['window_hours'] == window]

    pooled_rows = []
    for keys, gdf in hourly_all.groupby(group_cols):
        hour, traj, surv = keys
        row = {'hour': hour, 'trajectory': traj, 'survival_status': surv,
               'window_hours': window, 'n_sites': gdf['site'].nunique()}

        for vital in VITALS:
            mc, sc, nc = f'mean_{vital}', f'sd_{vital}', f'n_{vital}'
            if mc not in gdf.columns:
                continue
            sub = gdf[[mc, sc, nc]].dropna()
            if len(sub) == 0:
                continue

            means = sub[mc].values.astype(float)
            sds = sub[sc].values.astype(float)
            ns = sub[nc].values.astype(float)
            N = ns.sum()
            if N == 0:
                continue

            pm = (ns * means).sum() / N
            within = ((ns - 1) * sds ** 2).sum()
            between = (ns * (means - pm) ** 2).sum()
            ps = np.sqrt((within + between) / max(N - 1, 1))

            row[mc] = round(pm, 2)
            row[sc] = round(ps, 2)
            row[f'se_{vital}'] = round(ps / np.sqrt(N), 3)
            row[nc] = int(N)
            row[f'n_sites_{vital}'] = len(sub)

        pooled_rows.append(row)

    df = pd.DataFrame(pooled_rows)
    return df.sort_values(['trajectory', 'survival_status', 'hour']).reset_index(drop=True)


# Run pooling
pooled_hv = {}
for window in WINDOWS:
    hv = data[f'hv_{window}']
    if len(hv) == 0:
        continue
    pooled = pool_hourly_vitals(hv, window)
    pooled.to_csv(OUTPUT_DIR / f'pooled_hourly_vitals_{window}h.csv', index=False)
    pooled_hv[window] = pooled
    print(f'[OK] pooled_hourly_vitals_{window}h.csv ({len(pooled)} rows)')
    
    # Show summary
    print(f'\n  Shape: {pooled.shape}')
    print(f'  Hours: {pooled["hour"].min():.0f} - {pooled["hour"].max():.0f}')
    print(f'  Trajectories: {sorted(pooled["trajectory"].unique())}')
    
    # Preview: Normothermic, Survivor, hour 0-3
    preview = pooled[(pooled['trajectory'] == 'Normothermic') & 
                     (pooled['survival_status'] == 'Survivor') & 
                     (pooled['hour'] <= 3)]
    if len(preview) > 0:
        print(f'\n  Preview (Normothermic, Survivor, hr 0-3):')
        for _, r in preview.iterrows():
            print(f'    hr={r["hour"]:.0f}: HR={r.get("mean_heart_rate","NA")}, '
                  f'Temp={r.get("mean_temp_c","NA")}, '
                  f'MAP={r.get("mean_map","NA")}, '
                  f'SpO2={r.get("mean_spo2","NA")}, '
                  f'n_sites={r.get("n_sites","?")}')
    print()

In [ ]:
# =============================================================
# CELL 7: SITE COMPARISON TABLE
# =============================================================
for window in WINDOWS:
    t1 = data[f't1_{window}']
    if len(t1) == 0:
        continue
    
    print(f'\n{"=" * 60}')
    print(f'  SITE COMPARISON ({window}h)')
    print(f'{"=" * 60}')
    
    rows = []
    for site in sorted(t1['site'].unique()):
        sd = t1[t1['site'] == site]
        
        def gv(group, var):
            match = sd[(sd['group'] == group) & (sd['variable'] == var)]
            return match['value'].iloc[0] if len(match) > 0 else np.nan
        
        row = {
            'site': site,
            'n': gv('Overall', 'n'),
            'age_mean': gv('Overall', 'age_mean'),
            'age_sd': gv('Overall', 'age_sd'),
            'male_pct': gv('Overall', 'sex_male_pct'),
            'hosp_los_median': gv('Overall', 'hosp_los_median'),
            'icu_los_median': gv('Overall', 'icu_los_median'),
            'mortality_pct': gv('Overall', 'mortality_pct'),
        }
        for traj in TRAJ_ORDER:
            n_traj = gv(traj, 'n')
            mort_traj = gv(traj, 'mortality_pct')
            tkey = traj.lower().replace(' ', '_')
            row[f'n_{tkey}'] = n_traj
            row[f'mort_{tkey}'] = mort_traj
        rows.append(row)
    
    comp = pd.DataFrame(rows)
    comp.to_csv(OUTPUT_DIR / f'site_comparison_{window}h.csv', index=False)
    print(f'\n[OK] site_comparison_{window}h.csv')
    
    # Display
    print(f'\n  {"Site":15s} {"N":>6s} {"Age":>8s} {"Male%":>6s} {"Mort%":>6s} | '
          f'{"Hypo":>5s} {"Norm":>5s} {"RD":>5s} {"PH":>5s}')
    print(f'  {"-"*15} {"-"*6} {"-"*8} {"-"*6} {"-"*6} | {"-"*5} {"-"*5} {"-"*5} {"-"*5}')
    for _, r in comp.iterrows():
        print(f'  {r["site"]:15s} {int(r["n"]):>6,} {r["age_mean"]:>5.1f}±{r["age_sd"]:>4.1f} '
              f'{r["male_pct"]:>5.1f} {r["mortality_pct"]:>5.1f} | '
              f'{int(r.get("n_group_1", 0)):>5,} {int(r.get("n_group_2", 0)):>5,} '
              f'{int(r.get("n_group_3", 0)):>5,} {int(r.get("n_group_4", 0)):>5,}')

        # Totals
        print(f'  {"TOTAL":15s} {int(comp["n"].sum()):>6,} '
              f'{"":>8s} {"":>6s} {"":>6s} | '
              f'{int(comp["n_group_1"].sum()):>5,} {int(comp["n_group_2"].sum()):>5,} '
              f'{int(comp["n_group_3"].sum()):>5,} {int(comp["n_group_4"].sum()):>5,}')

In [ ]:
# =============================================================
# CELL 8: FORMATTED POOLED TABLE 1 (TEXT)
# =============================================================
for window in WINDOWS:
    if window not in pooled_t1:
        continue
    pooled = pooled_t1[window]
    n_sites = data[f't1_{window}']['site'].nunique()
    
    def gv(group, var):
        match = pooled[(pooled['group'] == group) & (pooled['variable'] == var)]
        return match['value'].iloc[0] if len(match) > 0 else None
    
    def fmt_n(group):
        v = gv(group, 'n')
        return f'{int(v):,}' if v is not None else '—'
    
    def fmt_age(group):
        m, s = gv(group, 'age_mean'), gv(group, 'age_sd')
        return f'{m} +/- {s}' if m is not None else '—'
    
    def fmt_male(group):
        n, p = gv(group, 'sex_male_n'), gv(group, 'sex_male_pct')
        return f'{int(n):,} ({p}%)' if n is not None else '—'
    
    def fmt_los(group, prefix):
        m = gv(group, f'{prefix}_median_approx')
        q25 = gv(group, f'{prefix}_q25_approx')
        q75 = gv(group, f'{prefix}_q75_approx')
        if m is not None and q25 is not None and q75 is not None:
            return f'{m} ({q25}-{q75})'
        elif m is not None:
            return f'{m}'
        return '—'
    
    def fmt_mort(group):
        n, p = gv(group, 'mortality_n'), gv(group, 'mortality_pct')
        return f'{int(n):,} ({p}%)' if n is not None else '—'
    
    fpath = OUTPUT_DIR / f'pooled_table1_{window}h.txt'
    with open(fpath, 'w', encoding='utf-8') as f:
        f.write('=' * 75 + '\n')
        f.write(f'  POOLED TABLE 1: OHCA ICU COHORT ({window}h, {n_sites} sites)\n')
        f.write('=' * 75 + '\n\n')
        
        f.write(f'  {"Variable":<35s} {"Overall":>14s} {"Survivor":>14s} {"Non-Surv":>14s}\n')
        f.write(f'  {"-"*35} {"-"*14} {"-"*14} {"-"*14}\n')
        
        for label, func in [
            ('N', fmt_n),
            ('Age, mean +/- SD', fmt_age),
            ('Male, n (%)', fmt_male),
            ('Hosp LOS, median (IQR)', lambda g: fmt_los(g, 'hosp_los')),
            ('ICU LOS, median (IQR)', lambda g: fmt_los(g, 'icu_los')),
            ('Mortality, n (%)', fmt_mort),
        ]:
            f.write(f'  {label:<35s} {func("Overall"):>14s} '
                    f'{func("Survivor"):>14s} {func("Non-Survivor"):>14s}\n')
        
        # Race breakdown
        f.write(f'\n  Race:\n')
        race_vars = [v for v in pooled[pooled['group'] == 'Overall']['variable'].unique() 
                     if v.startswith('race_') and v.endswith('_n')]
        for rv in sorted(race_vars):
            race_name = rv.replace('race_', '').replace('_n', '').replace('_', ' ').title()
            for g in ['Overall', 'Survivor', 'Non-Survivor']:
                pass  # just get overall
            n_val = gv('Overall', rv)
            pct_val = gv('Overall', rv.replace('_n', '_pct'))
            if n_val is not None:
                f.write(f'    {race_name:<33s} {int(n_val):>5,} ({pct_val:.1f}%)\n')
        
        # By trajectory
        f.write(f'\n\n{"=" * 75}\n  BY TEMPERATURE TRAJECTORY\n{"=" * 75}\n\n')
        f.write(f'  {"Variable":<25s} {"Hypothermic":>14s} {"Normothermic":>14s} '
                f'{"Rapid Decl":>14s} {"Persist Hi":>14s}\n')
        f.write(f'  {"-"*25} {"-"*14} {"-"*14} {"-"*14} {"-"*14}\n')
        
        for label, func in [
            ('N', fmt_n),
            ('Age, mean +/- SD', fmt_age),
            ('Mortality, n (%)', fmt_mort),
            ('Hosp LOS, median', lambda g: fmt_los(g, 'hosp_los')),
            ('ICU LOS, median', lambda g: fmt_los(g, 'icu_los')),
        ]:
            vals = [func(t) for t in TRAJ_ORDER]
            f.write(f'  {label:<25s} {vals[0]:>14s} {vals[1]:>14s} {vals[2]:>14s} {vals[3]:>14s}\n')
    
    print(f'[OK] pooled_table1_{window}h.txt')
    
    # Display
    with open(fpath, 'r') as f:
        print(f.read())

In [ ]:
# =============================================================
# CELL 8: FORMATTED POOLED TABLE 1 (TEXT) + VERIFICATION
# =============================================================
for window in WINDOWS:
    if window not in pooled_t1:
        continue
    pooled = pooled_t1[window]
    t1_raw = data[f't1_{window}']
    n_sites = t1_raw['site'].nunique()
    
    # --- VERIFICATION: pooled N vs sum of site N ---
    print(f'\n{"=" * 60}')
    print(f'  VERIFICATION: {window}h')
    print(f'{"=" * 60}')
    
    for group in ['Overall', 'Survivor', 'Non-Survivor'] + TRAJ_ORDER:
        # Pooled N
        pooled_n = pooled[(pooled['group'] == group) & (pooled['variable'] == 'n')]
        pn = int(pooled_n['value'].iloc[0]) if len(pooled_n) > 0 else 0
        
        # Sum of site-level N
        site_ns = t1_raw[(t1_raw['group'] == group) & (t1_raw['variable'] == 'n')]
        site_sum = int(site_ns['value'].sum()) if len(site_ns) > 0 else 0
        n_contributing = len(site_ns)
        
        match = '✓' if pn == site_sum else '✗ MISMATCH'
        print(f'  {group:20s}: pooled={pn:>8,}  sum_sites={site_sum:>8,}  ({n_contributing} sites) {match}')
    
    # Per-site detail for Overall
    print(f'\n  Per-site N (Overall):')
    site_ns = t1_raw[(t1_raw['group'] == 'Overall') & (t1_raw['variable'] == 'n')].sort_values('site')
    for _, r in site_ns.iterrows():
        print(f'    {r["site"]:15s}: {int(r["value"]):>6,}')
    print(f'    {"TOTAL":15s}: {int(site_ns["value"].sum()):>6,}')
    
    # Per-site detail for each trajectory
    print(f'\n  Per-site N by Trajectory:')
    print(f'  {"Site":15s}', end='')
    for traj in TRAJ_ORDER:
        print(f' {traj:>15s}', end='')
    print(f' {"Total":>8s}')
    print(f'  {"-"*15}', end='')
    for _ in TRAJ_ORDER:
        print(f' {"-"*15}', end='')
    print(f' {"-"*8}')
    
    for site in sorted(t1_raw['site'].unique()):
        sd = t1_raw[t1_raw['site'] == site]
        print(f'  {site:15s}', end='')
        row_total = 0
        for traj in TRAJ_ORDER:
            n_val = sd[(sd['group'] == traj) & (sd['variable'] == 'n')]['value']
            n = int(n_val.iloc[0]) if len(n_val) > 0 else 0
            row_total += n
            print(f' {n:>15,}', end='')
        print(f' {row_total:>8,}')
    
    # Column totals
    print(f'  {"TOTAL":15s}', end='')
    grand = 0
    for traj in TRAJ_ORDER:
        col_sum = int(t1_raw[(t1_raw['group'] == traj) & (t1_raw['variable'] == 'n')]['value'].sum())
        grand += col_sum
        print(f' {col_sum:>15,}', end='')
    print(f' {grand:>8,}')

    # --- NOW WRITE THE FORMATTED TABLE ---
    def gv(group, var):
        match = pooled[(pooled['group'] == group) & (pooled['variable'] == var)]
        return match['value'].iloc[0] if len(match) > 0 else None
    
    def fmt_n(group):
        v = gv(group, 'n')
        return f'{int(v):,}' if v is not None else '—'
    
    def fmt_age(group):
        m, s = gv(group, 'age_mean'), gv(group, 'age_sd')
        return f'{m} +/- {s}' if m is not None else '—'
    
    def fmt_male(group):
        n, p = gv(group, 'sex_male_n'), gv(group, 'sex_male_pct')
        return f'{int(n):,} ({p}%)' if n is not None else '—'
    
    def fmt_los(group, prefix):
        m = gv(group, f'{prefix}_median_approx')
        q25 = gv(group, f'{prefix}_q25_approx')
        q75 = gv(group, f'{prefix}_q75_approx')
        if m is not None and q25 is not None and q75 is not None:
            return f'{m} ({q25}-{q75})'
        elif m is not None:
            return f'{m}'
        return '—'
    
    def fmt_mort(group):
        n, p = gv(group, 'mortality_n'), gv(group, 'mortality_pct')
        return f'{int(n):,} ({p}%)' if n is not None else '—'
    
    fpath = OUTPUT_DIR / f'pooled_table1_{window}h.txt'
    with open(fpath, 'w', encoding='utf-8') as f:
        f.write('=' * 75 + '\n')
        f.write(f'  POOLED TABLE 1: OHCA ICU COHORT ({window}h, {n_sites} sites)\n')
        f.write('=' * 75 + '\n\n')
        
        f.write(f'  {"Variable":<35s} {"Overall":>14s} {"Survivor":>14s} {"Non-Surv":>14s}\n')
        f.write(f'  {"-"*35} {"-"*14} {"-"*14} {"-"*14}\n')
        
        for label, func in [
            ('N', fmt_n),
            ('Age, mean +/- SD', fmt_age),
            ('Male, n (%)', fmt_male),
            ('Hosp LOS, median (IQR)', lambda g: fmt_los(g, 'hosp_los')),
            ('ICU LOS, median (IQR)', lambda g: fmt_los(g, 'icu_los')),
            ('Mortality, n (%)', fmt_mort),
        ]:
            f.write(f'  {label:<35s} {func("Overall"):>14s} '
                    f'{func("Survivor"):>14s} {func("Non-Survivor"):>14s}\n')
        
        f.write(f'\n  Race:\n')
        race_vars = [v for v in pooled[pooled['group'] == 'Overall']['variable'].unique() 
                     if v.startswith('race_') and v.endswith('_n')]
        for rv in sorted(race_vars):
            race_name = rv.replace('race_', '').replace('_n', '').replace('_', ' ').title()
            n_val = gv('Overall', rv)
            pct_val = gv('Overall', rv.replace('_n', '_pct'))
            if n_val is not None:
                f.write(f'    {race_name:<33s} {int(n_val):>5,} ({pct_val:.1f}%)\n')
        
        f.write(f'\n\n{"=" * 75}\n  BY TEMPERATURE TRAJECTORY\n{"=" * 75}\n\n')
        f.write(f'  {"Variable":<25s} {"Hypothermic":>14s} {"Normothermic":>14s} '
                f'{"Rapid Decl":>14s} {"Persist Hi":>14s}\n')
        f.write(f'  {"-"*25} {"-"*14} {"-"*14} {"-"*14} {"-"*14}\n')
        
        for label, func in [
            ('N', fmt_n),
            ('Age, mean +/- SD', fmt_age),
            ('Male, n (%)', fmt_male),
            ('Mortality, n (%)', fmt_mort),
            ('Hosp LOS, median', lambda g: fmt_los(g, 'hosp_los')),
            ('ICU LOS, median', lambda g: fmt_los(g, 'icu_los')),
        ]:
            vals = [func(t) for t in TRAJ_ORDER]
            f.write(f'  {label:<25s} {vals[0]:>14s} {vals[1]:>14s} {vals[2]:>14s} {vals[3]:>14s}\n')
    
    print(f'\n[OK] pooled_table1_{window}h.txt')
    
    with open(fpath, 'r') as f:
        print(f.read())

In [ ]:
# =============================================================
# CELL 9: POOLED VITALS PLOTS
# =============================================================
for window in WINDOWS:
    if window not in pooled_hv:
        continue
    pooled = pooled_hv[window]
    n_sites = pooled['n_sites'].max()
    
    # --- Helper: weighted average across groups per hour ---
    def wavg_vital(g, mc, sec, nc):
        if g[nc].sum() == 0:
            return pd.Series({mc: np.nan, sec: np.nan, nc: 0})
        wm = np.average(g[mc], weights=g[nc])
        total_n = g[nc].sum()
        pooled_se = np.sqrt(sum((g[sec] * g[nc])**2)) / total_n
        return pd.Series({mc: wm, sec: pooled_se, nc: total_n})
    
    # --- Plot 1: Pooled temp by trajectory ---
    fig, ax = plt.subplots(figsize=(10, 6))
    for traj, color in TRAJ_COLORS.items():
        sub = pooled[pooled['trajectory'] == traj].copy()
        sub = sub.dropna(subset=['mean_temp_c', 'n_temp_c'])
        sub = sub[sub['n_temp_c'] > 0]
        if len(sub) == 0: continue
        grouped = sub.groupby('hour').apply(
            lambda g: wavg_vital(g, 'mean_temp_c', 'se_temp_c', 'n_temp_c')).reset_index()
        n = int(grouped['n_temp_c'].max())
        ax.plot(grouped['hour'], grouped['mean_temp_c'], linewidth=2, color=color,
                label=f'{traj} (n≈{n:,})')
        ax.fill_between(grouped['hour'],
                        grouped['mean_temp_c'] - 1.96 * grouped['se_temp_c'],
                        grouped['mean_temp_c'] + 1.96 * grouped['se_temp_c'],
                        color=color, alpha=0.15)
    ax.set_xlabel('Hours from First Vital')
    ax.set_ylabel('Temperature (°C)')
    ax.set_title(f'Pooled Temperature by Trajectory ({window}h, {n_sites} sites)')
    ax.legend(); ax.grid(True, alpha=0.3); fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'pooled_temp_by_trajectory_{window}h.png', dpi=150, bbox_inches='tight')
    print(f'[OK] pooled_temp_by_trajectory_{window}h.png'); plt.show()
    
    # --- Plot 2: Pooled temp by survival ---
    fig, ax = plt.subplots(figsize=(10, 6))
    for status, color in SURV_COLORS.items():
        sub = pooled[pooled['survival_status'] == status].copy()
        sub = sub.dropna(subset=['mean_temp_c', 'n_temp_c'])
        sub = sub[sub['n_temp_c'] > 0]
        if len(sub) == 0: continue
        grouped = sub.groupby('hour').apply(
            lambda g: wavg_vital(g, 'mean_temp_c', 'se_temp_c', 'n_temp_c')).reset_index()
        n = int(grouped['n_temp_c'].max())
        ax.plot(grouped['hour'], grouped['mean_temp_c'], linewidth=2, color=color,
                label=f'{status} (n≈{n:,})')
        ax.fill_between(grouped['hour'],
                        grouped['mean_temp_c'] - 1.96 * grouped['se_temp_c'],
                        grouped['mean_temp_c'] + 1.96 * grouped['se_temp_c'],
                        color=color, alpha=0.15)
    ax.set_xlabel('Hours from First Vital')
    ax.set_ylabel('Temperature (°C)')
    ax.set_title(f'Pooled Temperature: Survivors vs Non-Survivors ({window}h, {n_sites} sites)')
    ax.legend(); ax.grid(True, alpha=0.3); fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'pooled_temp_survival_{window}h.png', dpi=150, bbox_inches='tight')
    print(f'[OK] pooled_temp_survival_{window}h.png'); plt.show()
    
    # --- Plot 3: 2×2 all vitals by survival (weighted) ---
    vitals_plot = [
        ('heart_rate', 'Heart Rate (bpm)'),
        ('temp_c', 'Temperature (°C)'),
        ('map', 'MAP (mmHg)'),
        ('spo2', 'SpO₂ (%)'),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    for ax, (vital, ylabel) in zip(axes.flatten(), vitals_plot):
        mc, sec, nc = f'mean_{vital}', f'se_{vital}', f'n_{vital}'
        if mc not in pooled.columns: continue
        for status, color in SURV_COLORS.items():
            sub = pooled[pooled['survival_status'] == status].copy()
            sub = sub.dropna(subset=[mc, nc])
            sub = sub[sub[nc] > 0]
            if len(sub) == 0: continue
            grouped = sub.groupby('hour').apply(
                lambda g: wavg_vital(g, mc, sec, nc)).reset_index()
            grouped = grouped.dropna(subset=[mc])
            ax.plot(grouped['hour'], grouped[mc], linewidth=1.5, color=color, label=status)
            if sec in grouped.columns:
                ax.fill_between(grouped['hour'], grouped[mc] - 1.96 * grouped[sec],
                                grouped[mc] + 1.96 * grouped[sec], color=color, alpha=0.15)
        ax.set_xlabel('Hours from First Vital'); ax.set_ylabel(ylabel)
        ax.set_title(ylabel); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    fig.suptitle(f'Pooled Vitals: Survivors vs Non-Survivors ({window}h, {n_sites} sites)',
                 fontsize=13, fontweight='bold', y=1.02)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'pooled_vitals_survival_{window}h.png', dpi=150, bbox_inches='tight')
    print(f'[OK] pooled_vitals_survival_{window}h.png'); plt.show()
    
    # --- Plot 4: 2×2 all vitals by trajectory (weighted) ---
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    for ax, (vital, ylabel) in zip(axes.flatten(), vitals_plot):
        mc, sec, nc = f'mean_{vital}', f'se_{vital}', f'n_{vital}'
        if mc not in pooled.columns: continue
        for traj, color in TRAJ_COLORS.items():
            sub = pooled[pooled['trajectory'] == traj].copy()
            sub = sub.dropna(subset=[mc, nc])
            sub = sub[sub[nc] > 0]
            if len(sub) == 0: continue
            grouped = sub.groupby('hour').apply(
                lambda g: wavg_vital(g, mc, sec, nc)).reset_index()
            grouped = grouped.dropna(subset=[mc])
            ax.plot(grouped['hour'], grouped[mc], linewidth=1.5, color=color, label=traj)
            if sec in grouped.columns:
                ax.fill_between(grouped['hour'], grouped[mc] - 1.96 * grouped[sec],
                                grouped[mc] + 1.96 * grouped[sec], color=color, alpha=0.1)
        ax.set_xlabel('Hours from First Vital'); ax.set_ylabel(ylabel)
        ax.set_title(ylabel); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
    fig.suptitle(f'Pooled Vitals by Temperature Trajectory ({window}h, {n_sites} sites)',
                 fontsize=13, fontweight='bold', y=1.02)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'pooled_vitals_by_trajectory_{window}h.png', dpi=150, bbox_inches='tight')
    print(f'[OK] pooled_vitals_by_trajectory_{window}h.png'); plt.show()

In [ ]:
# =============================================================
# CELL 10: POOLED MORTALITY BAR CHART
# =============================================================
for window in WINDOWS:
    if window not in pooled_t1:
        continue
    pooled = pooled_t1[window]
    n_sites = data[f't1_{window}']['site'].nunique()
    
    fig, ax = plt.subplots(figsize=(8, 5))
    mort_rates, totals = [], []
    for traj in TRAJ_ORDER:
        td = pooled[pooled['group'] == traj]
        n_val = td[td['variable'] == 'n']['value']
        mort_val = td[td['variable'] == 'mortality_pct']['value']
        n = int(n_val.iloc[0]) if len(n_val) > 0 else 0
        m = float(mort_val.iloc[0]) if len(mort_val) > 0 else 0
        totals.append(n)
        mort_rates.append(m)
    
    bars = ax.bar(TRAJ_ORDER, mort_rates, color=[TRAJ_COLORS[t] for t in TRAJ_ORDER])
    for bar, rate, n in zip(bars, mort_rates, totals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f'{rate:.1f}%\n(n={n:,})', ha='center', fontsize=9)
    ax.set_ylabel('Mortality (%)')
    ax.set_title(f'Pooled Mortality by Trajectory ({window}h, {n_sites} sites)')
    ax.set_ylim(0, max(mort_rates) + 15)
    ax.grid(True, alpha=0.3, axis='y')
    plt.xticks(rotation=15, ha='right')
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'pooled_mortality_by_trajectory_{window}h.png', dpi=150, bbox_inches='tight')
    print(f'[OK] pooled_mortality_by_trajectory_{window}h.png')
    plt.show()

In [ ]:
# =============================================================
# SITE OVERLAY: 2×2 ALL VITALS — ALL TRAJECTORIES IN ONE PLOT
# =============================================================
vitals_plot = [
    ('heart_rate', 'Heart Rate (bpm)'),
    ('temp_c', 'Temperature (°C)'),
    ('map', 'MAP (mmHg)'),
    ('spo2', 'SpO₂ (%)'),
]

for window in WINDOWS:
    hv = data[f'hv_{window}']
    if len(hv) == 0:
        continue
    
    sites = sorted(hv['site'].unique())
    n_sites = len(sites)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    for ax, (vital, ylabel) in zip(axes.flatten(), vitals_plot):
        mc, nc, sec = f'mean_{vital}', f'n_{vital}', f'se_{vital}'
        if mc not in hv.columns:
            continue
        
        # Thin site lines — colored by trajectory
        for site in sites:
            for traj in TRAJ_ORDER:
                sub = hv[(hv['site'] == site) & (hv['trajectory'] == traj)].copy()
                sub = sub.dropna(subset=[mc, nc])
                sub = sub[sub[nc] > 0]
                if len(sub) == 0:
                    continue
                
                def wavg_v(g):
                    if g[nc].sum() == 0:
                        return pd.Series({'mean_val': np.nan, 'total_n': 0})
                    wm = np.average(g[mc], weights=g[nc])
                    return pd.Series({'mean_val': wm, 'total_n': g[nc].sum()})
                
                grouped = sub.groupby('hour').apply(wavg_v).reset_index()
                grouped = grouped.dropna(subset=['mean_val'])
                grouped = grouped[grouped['total_n'] >= 5]
                if len(grouped) == 0:
                    continue
                
                ax.plot(grouped['hour'], grouped['mean_val'],
                        linewidth=0.5, color=TRAJ_COLORS[traj], alpha=0.25)
        
        # Thick pooled lines — colored by trajectory
        if window in pooled_hv:
            for traj in TRAJ_ORDER:
                pooled_sub = pooled_hv[window][pooled_hv[window]['trajectory'] == traj].copy()
                pooled_sub = pooled_sub.dropna(subset=[mc, nc])
                pooled_sub = pooled_sub[pooled_sub[nc] > 0]
                if len(pooled_sub) == 0:
                    continue
                
                def wavg_p(g):
                    if g[nc].sum() == 0:
                        return pd.Series({mc: np.nan, sec: np.nan})
                    wm = np.average(g[mc], weights=g[nc])
                    total_n = g[nc].sum()
                    pooled_se = np.sqrt(sum((g[sec] * g[nc])**2)) / total_n
                    return pd.Series({mc: wm, sec: pooled_se})
                
                pg = pooled_sub.groupby('hour').apply(wavg_p).reset_index()
                ax.plot(pg['hour'], pg[mc], linewidth=2.5, color=TRAJ_COLORS[traj],
                        alpha=1.0, zorder=10)
                ax.fill_between(pg['hour'],
                                pg[mc] - 1.96 * pg[sec],
                                pg[mc] + 1.96 * pg[sec],
                                color=TRAJ_COLORS[traj], alpha=0.08, zorder=9)
        
        ax.set_xlabel('Hours from First Vital')
        ax.set_ylabel(ylabel)
        ax.set_title(ylabel, fontsize=11, fontweight='bold')
        ax.grid(True, alpha=0.3)
    
    # Single legend — trajectory colors
    from matplotlib.lines import Line2D
    handles = [Line2D([0], [0], color=TRAJ_COLORS[t], linewidth=2.5, label=t) for t in TRAJ_ORDER]
    handles.append(Line2D([0], [0], color='gray', linewidth=0.5, alpha=0.4, label='Individual sites'))
    axes[0][1].legend(handles=handles, fontsize=8, loc='upper right')
    
    fig.suptitle(f'Pooled Vitals by Trajectory with Site Overlay ({window}h, {n_sites} CLIF Sites)\n'
                 f'Thin faded = individual sites | Thick = pooled estimate',
                 fontsize=13, fontweight='bold', y=1.03)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'site_overlay_vitals_all_{window}h.png', dpi=200, bbox_inches='tight')
    print(f'[OK] site_overlay_vitals_all_{window}h.png')
    plt.show()

In [ ]:
# =============================================================
# CELL 11: SUMMARY
# =============================================================
print('=' * 60)
print('  CONSOLIDATION COMPLETE')
print('=' * 60)
print(f'\n  Output: {OUTPUT_DIR}')
print(f'\n  Files generated:')
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f'    {f.name:50s} {f.stat().st_size / 1024:>8.1f} KB')
print(f'\n  Sites: {len(site_dirs)}')
for window in WINDOWS:
    if window in pooled_t1:
        overall = pooled_t1[window]
        n = overall[(overall['group'] == 'Overall') & (overall['variable'] == 'n')]['value']
        mort = overall[(overall['group'] == 'Overall') & (overall['variable'] == 'mortality_pct')]['value']
        print(f'  {window}h: N={int(n.iloc[0]):,}, Mortality={mort.iloc[0]:.1f}%')

In [ ]:
# =============================================================
# FIGURE 1: FOREST PLOT — MORTALITY BY TRAJECTORY (SITE-LEVEL)
# =============================================================
# This is the key figure for multi-site studies.
# Shows each site's mortality estimate with CI + pooled diamond.

from scipy import stats

for window in WINDOWS:
    t1 = data[f't1_{window}']
    if len(t1) == 0:
        continue
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 8), sharey=True)
    sites = sorted(t1['site'].unique())
    
    for ax, traj in zip(axes, TRAJ_ORDER):
        site_data = []
        for site in sites:
            sd = t1[(t1['site'] == site) & (t1['group'] == traj)]
            n_val = sd[sd['variable'] == 'n']['value']
            mort_n = sd[sd['variable'] == 'mortality_n']['value']
            if len(n_val) > 0 and len(mort_n) > 0:
                n = int(n_val.iloc[0])
                d = int(mort_n.iloc[0])
                if n > 0:
                    p = d / n
                    # Wilson score CI
                    if n >= 5:
                        z = 1.96
                        denom = 1 + z**2 / n
                        center = (p + z**2 / (2*n)) / denom
                        margin = z * np.sqrt((p*(1-p) + z**2/(4*n)) / n) / denom
                        ci_lo = max(0, center - margin) * 100
                        ci_hi = min(1, center + margin) * 100
                    else:
                        ci_lo, ci_hi = 0, 100
                    site_data.append({'site': site, 'n': n, 'deaths': d,
                                      'mort': p*100, 'ci_lo': ci_lo, 'ci_hi': ci_hi})
        
        if not site_data:
            ax.set_title(traj)
            continue
        
        sdf = pd.DataFrame(site_data)
        
        # Plot site estimates
        y_pos = list(range(len(sdf)))
        for i, (_, r) in enumerate(sdf.iterrows()):
            marker_size = max(4, min(12, r['n'] / 50))  # size by N
            ax.plot(r['mort'], i, 'o', color=TRAJ_COLORS[traj], 
                    markersize=marker_size, zorder=3)
            ax.plot([r['ci_lo'], r['ci_hi']], [i, i], '-', 
                    color=TRAJ_COLORS[traj], linewidth=1.5, zorder=2)
            ax.text(107, i, f' n={r["n"]}', va='center', fontsize=7, color='gray')
        
        # Pooled estimate (weighted average)
        total_n = sdf['n'].sum()
        total_d = sdf['deaths'].sum()
        pooled_mort = total_d / total_n * 100
        # Pooled CI
        p_pool = total_d / total_n
        z = 1.96
        denom = 1 + z**2 / total_n
        center = (p_pool + z**2 / (2*total_n)) / denom
        margin = z * np.sqrt((p_pool*(1-p_pool) + z**2/(4*total_n)) / total_n) / denom
        pool_lo = max(0, center - margin) * 100
        pool_hi = min(1, center + margin) * 100
        
        # Diamond for pooled
        diamond_y = -1.5
        diamond_h = 0.4
        diamond = plt.Polygon([
            (pool_lo, diamond_y), (pooled_mort, diamond_y + diamond_h),
            (pool_hi, diamond_y), (pooled_mort, diamond_y - diamond_h)
        ], color=TRAJ_COLORS[traj], alpha=0.8, zorder=3)
        ax.add_patch(diamond)
        ax.axvline(pooled_mort, color=TRAJ_COLORS[traj], linestyle='--', alpha=0.3)
        
        # Labels
        ax.set_yticks(y_pos + [-1.5])
        ax.set_yticklabels(sdf['site'].tolist() + [f'Pooled\n(n={total_n:,})'], fontsize=8)
        ax.set_xlabel('Mortality (%)')
        ax.set_title(f'{traj}\n{pooled_mort:.1f}% ({pool_lo:.1f}-{pool_hi:.1f})', 
                     fontsize=11, fontweight='bold', color=TRAJ_COLORS[traj])
        ax.set_xlim(-5, 105)
        ax.grid(True, alpha=0.2, axis='x')
        ax.axhline(-0.75, color='black', linewidth=0.5)
    
    fig.suptitle(f'Site-Level Mortality by Temperature Trajectory ({window}h, {len(sites)} sites)',
                 fontsize=14, fontweight='bold', y=1.02)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'forest_plot_mortality_{window}h.png', dpi=200, bbox_inches='tight')
    print(f'[OK] forest_plot_mortality_{window}h.png')
    plt.show()

In [ ]:
# =============================================================
# FIGURE 2: HEATMAP — SITE × TRAJECTORY MORTALITY
# =============================================================
import matplotlib.colors as mcolors

for window in WINDOWS:
    t1 = data[f't1_{window}']
    if len(t1) == 0:
        continue
    
    sites = sorted(t1['site'].unique())
    
    # Build mortality matrix
    mort_matrix = np.zeros((len(sites), len(TRAJ_ORDER)))
    n_matrix = np.zeros((len(sites), len(TRAJ_ORDER)))
    
    for i, site in enumerate(sites):
        for j, traj in enumerate(TRAJ_ORDER):
            sd = t1[(t1['site'] == site) & (t1['group'] == traj)]
            mort = sd[sd['variable'] == 'mortality_pct']['value']
            n_val = sd[sd['variable'] == 'n']['value']
            mort_matrix[i, j] = mort.iloc[0] if len(mort) > 0 else np.nan
            n_matrix[i, j] = int(n_val.iloc[0]) if len(n_val) > 0 else 0
    
    # Add pooled row
    pooled = pooled_t1[window]
    pooled_mort = []
    pooled_ns = []
    for traj in TRAJ_ORDER:
        td = pooled[pooled['group'] == traj]
        m = td[td['variable'] == 'mortality_pct']['value']
        n = td[td['variable'] == 'n']['value']
        pooled_mort.append(float(m.iloc[0]) if len(m) > 0 else np.nan)
        pooled_ns.append(int(n.iloc[0]) if len(n) > 0 else 0)
    
    mort_matrix = np.vstack([mort_matrix, [pooled_mort]])
    n_matrix = np.vstack([n_matrix, [pooled_ns]])
    all_labels = sites + ['POOLED']
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 9))
    im = ax.imshow(mort_matrix, cmap='RdYlGn_r', vmin=0, vmax=100, aspect='auto')
    
    for i in range(len(all_labels)):
        for j in range(len(TRAJ_ORDER)):
            val = mort_matrix[i, j]
            n = int(n_matrix[i, j])
            if not np.isnan(val):
                text_color = 'white' if val > 60 else 'black'
                fs = 9 if i == len(all_labels) - 1 else 8
                ax.text(j, i, f'{val:.1f}%\n(n={n:,})', ha='center', va='center',
                        fontsize=fs, fontweight='bold', color=text_color)
    
    ax.set_xticks(range(len(TRAJ_ORDER)))
    ax.set_xticklabels(TRAJ_ORDER, rotation=20, ha='right', fontsize=10)
    ax.set_yticks(range(len(all_labels)))
    ax.set_yticklabels(all_labels, fontsize=10)
    # Bold the POOLED label
    ax.get_yticklabels()[-1].set_fontweight('bold')
    
    ax.axhline(len(sites) - 0.5, color='black', linewidth=2)
    
    cbar = plt.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label('Mortality (%)', fontsize=11)
    
    ax.set_title(f'Mortality by Site × Trajectory ({window}h, {len(sites)} sites)',
                 fontsize=13, fontweight='bold', pad=15)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'heatmap_mortality_{window}h.png', dpi=200, bbox_inches='tight')
    print(f'[OK] heatmap_mortality_{window}h.png')
    plt.show()

In [ ]:
# =============================================================
# FIGURE 3: SITE OVERVIEW — N + MORTALITY + TRAJECTORY MIX
# =============================================================
for window in WINDOWS:
    t1 = data[f't1_{window}']
    if len(t1) == 0:
        continue
    
    sites = sorted(t1['site'].unique())
    
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 7))
    
    # --- Panel A: Cohort size ---
    site_ns = []
    for site in sites:
        n = t1[(t1['site'] == site) & (t1['group'] == 'Overall') & (t1['variable'] == 'n')]['value']
        site_ns.append(int(n.iloc[0]) if len(n) > 0 else 0)
    
    bars = ax1.barh(sites, site_ns, color='#42A5F5', edgecolor='white')
    for bar, n in zip(bars, site_ns):
        ax1.text(bar.get_width() + max(site_ns) * 0.02, bar.get_y() + bar.get_height()/2,
                 f'{n:,}', va='center', fontsize=9)
    ax1.set_xlabel('Number of Patients')
    ax1.set_title('A. Cohort Size', fontsize=12, fontweight='bold')
    ax1.grid(True, alpha=0.2, axis='x')
    ax1.set_xlim(0, max(site_ns) * 1.25)
    
    # --- Panel B: Overall mortality ---
    site_morts = []
    for site in sites:
        m = t1[(t1['site'] == site) & (t1['group'] == 'Overall') & (t1['variable'] == 'mortality_pct')]['value']
        site_morts.append(float(m.iloc[0]) if len(m) > 0 else 0)
    
    colors_mort = ['#E53935' if m > 50 else '#FF8F00' if m > 40 else '#43A047' for m in site_morts]
    bars = ax2.barh(sites, site_morts, color=colors_mort, edgecolor='white')
    for bar, m in zip(bars, site_morts):
        ax2.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                 f'{m:.1f}%', va='center', fontsize=9)
    # Pooled line
    pooled_mort = pooled_t1[window]
    pm = pooled_mort[(pooled_mort['group'] == 'Overall') & (pooled_mort['variable'] == 'mortality_pct')]['value']
    if len(pm) > 0:
        ax2.axvline(pm.iloc[0], color='black', linestyle='--', linewidth=1.5, label=f'Pooled ({pm.iloc[0]:.1f}%)')
        ax2.legend(fontsize=8)
    ax2.set_xlabel('Mortality (%)')
    ax2.set_title('B. Overall Mortality', fontsize=12, fontweight='bold')
    ax2.set_xlim(0, 100)
    ax2.grid(True, alpha=0.2, axis='x')
    
    # --- Panel C: Trajectory distribution (stacked bar) ---
    bottom = np.zeros(len(sites))
    for traj in TRAJ_ORDER:
        pcts = []
        for site in sites:
            sd = t1[(t1['site'] == site)]
            n_traj = sd[(sd['group'] == traj) & (sd['variable'] == 'n')]['value']
            n_overall = sd[(sd['group'] == 'Overall') & (sd['variable'] == 'n')]['value']
            if len(n_traj) > 0 and len(n_overall) > 0 and n_overall.iloc[0] > 0:
                pcts.append(n_traj.iloc[0] / n_overall.iloc[0] * 100)
            else:
                pcts.append(0)
        ax3.barh(sites, pcts, left=bottom, color=TRAJ_COLORS[traj], 
                 edgecolor='white', label=traj)
        bottom += np.array(pcts)
    
    ax3.set_xlabel('Trajectory Distribution (%)')
    ax3.set_title('C. Trajectory Mix', fontsize=12, fontweight='bold')
    ax3.legend(fontsize=7, loc='lower right')
    ax3.set_xlim(0, 105)
    ax3.grid(True, alpha=0.2, axis='x')
    
    fig.suptitle(f'Multi-Site Overview ({window}h, {len(sites)} CLIF Sites)',
                 fontsize=14, fontweight='bold', y=1.02)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'site_overview_{window}h.png', dpi=200, bbox_inches='tight')
    print(f'[OK] site_overview_{window}h.png')
    plt.show()

In [ ]:
# =============================================================
# FIGURE 4: POOLED TEMP CURVES — TRAJECTORY × SURVIVAL (FACETED)
# =============================================================
for window in WINDOWS:
    if window not in pooled_hv:
        continue
    pooled = pooled_hv[window]
    n_sites = pooled['n_sites'].max()
    
    if 'mean_temp_c' not in pooled.columns:
        continue
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    for idx, traj in enumerate(TRAJ_ORDER):
        ax = axes[idx // 2][idx % 2]
        for status, color in SURV_COLORS.items():
            sub = pooled[(pooled['trajectory'] == traj) & (pooled['survival_status'] == status)]
            if len(sub) == 0 or sub['mean_temp_c'].isna().all():
                continue
            sub = sub.sort_values('hour')
            n_max = int(sub['n_temp_c'].max()) if 'n_temp_c' in sub.columns else 0
            ax.plot(sub['hour'], sub['mean_temp_c'], linewidth=1.5, color=color,
                    label=f'{status} (n≈{n_max:,})')
            if 'se_temp_c' in sub.columns:
                ax.fill_between(sub['hour'],
                                sub['mean_temp_c'] - 1.96 * sub['se_temp_c'],
                                sub['mean_temp_c'] + 1.96 * sub['se_temp_c'],
                                color=color, alpha=0.15)
        ax.set_xlabel('Hours from First Vital')
        ax.set_ylabel('Temperature (°C)')
        ax.set_title(traj, fontsize=12, fontweight='bold', color=TRAJ_COLORS[traj])
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    
    fig.suptitle(f'Pooled Temperature: Survivor vs Non-Survivor by Trajectory\n'
                 f'({window}h, {n_sites} CLIF sites)',
                 fontsize=14, fontweight='bold', y=1.04)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'pooled_temp_traj_survival_facet_{window}h.png', dpi=200, bbox_inches='tight')
    print(f'[OK] pooled_temp_traj_survival_facet_{window}h.png')
    plt.show()

In [ ]:
# =============================================================
# FIGURE 5: POOLED 2×2 ALL VITALS BY TRAJECTORY
# =============================================================
vitals_plot_info = [
    ('heart_rate', 'Heart Rate (bpm)'),
    ('temp_c', 'Temperature (°C)'),
    ('map', 'MAP (mmHg)'),
    ('spo2', 'SpO₂ (%)'),
]

for window in WINDOWS:
    if window not in pooled_hv:
        continue
    pooled = pooled_hv[window]
    n_sites = pooled['n_sites'].max()
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    for ax, (vital, ylabel) in zip(axes.flatten(), vitals_plot_info):
        mc = f'mean_{vital}'
        sec = f'se_{vital}'
        nc = f'n_{vital}'
        if mc not in pooled.columns:
            continue
        for traj, color in TRAJ_COLORS.items():
            sub = pooled[pooled['trajectory'] == traj].copy()
            sub = sub.dropna(subset=[mc, nc])
            sub = sub[sub[nc] > 0]
            if len(sub) == 0:
                continue
            
            def wavg(g):
                if g[nc].sum() == 0:
                    return pd.Series({mc: np.nan, sec: np.nan, nc: 0})
                wm = np.average(g[mc], weights=g[nc])
                total_n = g[nc].sum()
                pooled_se = np.sqrt(sum((g[sec] * g[nc])**2)) / total_n
                return pd.Series({mc: wm, sec: pooled_se, nc: total_n})
            
            grouped = sub.groupby('hour').apply(wavg).reset_index()
            grouped = grouped.dropna(subset=[mc])
            
            ax.plot(grouped['hour'], grouped[mc], linewidth=1.5, color=color, label=traj)
            if sec in grouped.columns:
                ax.fill_between(grouped['hour'], grouped[mc] - 1.96 * grouped[sec],
                                grouped[mc] + 1.96 * grouped[sec], color=color, alpha=0.1)
        ax.set_xlabel('Hours')
        ax.set_ylabel(ylabel)
        ax.set_title(ylabel, fontsize=11, fontweight='bold')
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)
    
    fig.suptitle(f'Pooled Vitals by Temperature Trajectory ({window}h, {n_sites} sites)',
                 fontsize=14, fontweight='bold', y=1.02)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'pooled_vitals_by_trajectory_{window}h.png', dpi=200, bbox_inches='tight')
    print(f'[OK] pooled_vitals_by_trajectory_{window}h.png')
    plt.show()

In [ ]:
# =============================================================
# FIGURE 6: FUNNEL PLOT — SITE MORTALITY VS SAMPLE SIZE
# =============================================================
for window in WINDOWS:
    t1 = data[f't1_{window}']
    if len(t1) == 0:
        continue
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Panel A: Overall mortality funnel
    ax = axes[0]
    sites = sorted(t1['site'].unique())
    ns, morts = [], []
    for site in sites:
        sd = t1[(t1['site'] == site) & (t1['group'] == 'Overall')]
        n = sd[sd['variable'] == 'n']['value']
        m = sd[sd['variable'] == 'mortality_pct']['value']
        if len(n) > 0 and len(m) > 0:
            ns.append(int(n.iloc[0]))
            morts.append(float(m.iloc[0]))
    
    pooled_m = sum(m * n for m, n in zip(morts, ns)) / sum(ns)
    
    # 95% and 99% CI funnels
    x_range = np.linspace(min(ns) * 0.5, max(ns) * 1.5, 100)
    for z, ls, label in [(1.96, '--', '95% CI'), (2.576, ':', '99% CI')]:
        se = np.sqrt(pooled_m * (100 - pooled_m) / x_range)
        ax.plot(x_range, pooled_m + z * se, ls, color='gray', alpha=0.5)
        ax.plot(x_range, pooled_m - z * se, ls, color='gray', alpha=0.5, label=label)
    
    ax.axhline(pooled_m, color='red', linestyle='-', linewidth=1, alpha=0.5,
               label=f'Pooled: {pooled_m:.1f}%')
    
    for site, n, m in zip(sites, ns, morts):
        ax.scatter(n, m, s=60, color='#1565C0', zorder=3)
        ax.annotate(site, (n, m), textcoords='offset points', xytext=(5, 5),
                    fontsize=7, alpha=0.8)
    
    ax.set_xlabel('Sample Size (N)')
    ax.set_ylabel('Mortality (%)')
    ax.set_title('A. Overall Mortality', fontsize=12, fontweight='bold')
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(True, alpha=0.2)
    
    # Panel B: Hypothermic mortality funnel
    # Panel B: Group 1 mortality funnel
    ax = axes[1]
    ns_h, morts_h, sites_h = [], [], []
    for site in sites:
        sd = t1[(t1['site'] == site) & (t1['group'] == 'Group 1')]
        n = sd[sd['variable'] == 'n']['value']
        m = sd[sd['variable'] == 'mortality_pct']['value']
        if len(n) > 0 and len(m) > 0 and int(n.iloc[0]) > 0:
            ns_h.append(int(n.iloc[0]))
            morts_h.append(float(m.iloc[0]))
            sites_h.append(site)
    
    if ns_h:
        pooled_h = sum(m * n for m, n in zip(morts_h, ns_h)) / sum(ns_h)
        x_range = np.linspace(max(1, min(ns_h) * 0.5), max(ns_h) * 1.5, 100)
        for z, ls, label in [(1.96, '--', '95% CI'), (2.576, ':', '99% CI')]:
            se = np.sqrt(pooled_h * (100 - pooled_h) / x_range)
            ax.plot(x_range, pooled_h + z * se, ls, color='gray', alpha=0.5)
            ax.plot(x_range, pooled_h - z * se, ls, color='gray', alpha=0.5, label=label)
        ax.axhline(pooled_h, color='red', linestyle='-', linewidth=1, alpha=0.5,
                   label=f'Pooled: {pooled_h:.1f}%')
        for site, n, m in zip(sites_h, ns_h, morts_h):
            ax.scatter(n, m, s=60, color=TRAJ_COLORS['Group 1'], zorder=3)
            ax.annotate(site, (n, m), textcoords='offset points', xytext=(5, 5),
                        fontsize=7, alpha=0.8)
    
    ax.set_xlabel('Sample Size (N)')
    ax.set_ylabel('Mortality (%)')
    ax.set_title('B. Group 1 Trajectory Mortality', fontsize=12, fontweight='bold')
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(True, alpha=0.2)
    
    fig.suptitle(f'Funnel Plots — Site Heterogeneity Check ({window}h)',
                 fontsize=14, fontweight='bold', y=1.02)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'funnel_plot_{window}h.png', dpi=200, bbox_inches='tight')
    print(f'[OK] funnel_plot_{window}h.png')
    plt.show()

In [ ]:
# =============================================================
# FIGURE 7: GRANT SUMMARY — SINGLE COMPOSITE FIGURE (IMPROVED)
# =============================================================
window = 72
if window in pooled_t1 and window in pooled_hv:
    t1 = data[f't1_{window}']
    pooled = pooled_hv[window]
    pt1 = pooled_t1[window]
    n_sites = t1['site'].nunique()
    
    fig = plt.figure(figsize=(20, 14))
    gs = fig.add_gridspec(2, 4, hspace=0.35, wspace=0.35)
    
    # ========= TOP ROW: THE STORY =========
    
    # --- A: Site scatter (scope of study) ---
    ax = fig.add_subplot(gs[0, 0])
    sites = sorted(t1['site'].unique())
    site_ns, site_morts = [], []
    for site in sites:
        sd = t1[(t1['site'] == site) & (t1['group'] == 'Overall')]
        n = sd[sd['variable'] == 'n']['value']
        m = sd[sd['variable'] == 'mortality_pct']['value']
        site_ns.append(int(n.iloc[0]) if len(n) > 0 else 0)
        site_morts.append(float(m.iloc[0]) if len(m) > 0 else 0)
    ax.scatter(site_ns, site_morts, s=80, c='#1565C0', edgecolors='white', zorder=3)
    for site, n, m in zip(sites, site_ns, site_morts):
        ax.annotate(site, (n, m), textcoords='offset points', xytext=(5, 5), fontsize=7)
    total_n = sum(site_ns)
    ax.set_xlabel('Cohort Size')
    ax.set_ylabel('Mortality (%)')
    ax.set_title(f'A. {n_sites} CLIF Sites (N={total_n:,})', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.2)
    
    # --- B: Pooled temp by trajectory (weighted) ---
    ax = fig.add_subplot(gs[0, 1:3])  # span 2 columns
    if 'mean_temp_c' in pooled.columns:
        for traj, color in TRAJ_COLORS.items():
            sub = pooled[pooled['trajectory'] == traj].copy()
            sub = sub.dropna(subset=['mean_temp_c', 'n_temp_c'])
            sub = sub[sub['n_temp_c'] > 0]
            
            def wavg(g):
                if g['n_temp_c'].sum() == 0:
                    return pd.Series({'mean_temp_c': np.nan, 'se_temp_c': np.nan, 'n_temp_c': 0})
                wm = np.average(g['mean_temp_c'], weights=g['n_temp_c'])
                total_n = g['n_temp_c'].sum()
                pooled_se = np.sqrt(sum((g['se_temp_c'] * g['n_temp_c'])**2)) / total_n
                return pd.Series({'mean_temp_c': wm, 'se_temp_c': pooled_se, 'n_temp_c': total_n})
            
            grouped = sub.groupby('hour').apply(wavg).reset_index()
            n = int(grouped['n_temp_c'].max())
            ax.plot(grouped['hour'], grouped['mean_temp_c'], linewidth=2, color=color,
                    label=f'{traj} (n≈{n:,})')
            ax.fill_between(grouped['hour'],
                            grouped['mean_temp_c'] - 1.96 * grouped['se_temp_c'],
                            grouped['mean_temp_c'] + 1.96 * grouped['se_temp_c'],
                            color=color, alpha=0.12)
    ax.set_xlabel('Hours from First Vital')
    ax.set_ylabel('Temperature (°C)')
    ax.set_title('B. Four Distinct Temperature Trajectories', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    # --- C: Pooled mortality bar ---
    ax = fig.add_subplot(gs[0, 3])
    mort_rates, totals = [], []
    for traj in TRAJ_ORDER:
        td = pt1[pt1['group'] == traj]
        n = td[td['variable'] == 'n']['value']
        m = td[td['variable'] == 'mortality_pct']['value']
        totals.append(int(n.iloc[0]) if len(n) > 0 else 0)
        mort_rates.append(float(m.iloc[0]) if len(m) > 0 else 0)
    bars = ax.bar(range(len(TRAJ_ORDER)), mort_rates, color=[TRAJ_COLORS[t] for t in TRAJ_ORDER])
    for bar, rate, n in zip(bars, mort_rates, totals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
                f'{rate:.1f}%\n(n={n:,})', ha='center', fontsize=8)
    ax.set_xticks(range(len(TRAJ_ORDER)))
    ax.set_xticklabels([t.replace(' ', '\n') for t in TRAJ_ORDER], fontsize=7)
    ax.set_ylabel('Mortality (%)')
    ax.set_title('C. Mortality by Trajectory', fontsize=11, fontweight='bold')
    ax.set_ylim(0, max(mort_rates) + 15)
    ax.grid(True, alpha=0.2, axis='y')
    
    # ========= BOTTOM ROW: ALL 4 TRAJECTORIES — SURVIVOR VS NON-SURVIVOR =========
    for idx, traj in enumerate(TRAJ_ORDER):
        ax = fig.add_subplot(gs[1, idx])
        if 'mean_temp_c' in pooled.columns:
            for status, color in SURV_COLORS.items():
                sub = pooled[(pooled['trajectory'] == traj) & 
                             (pooled['survival_status'] == status)].sort_values('hour')
                if len(sub) == 0:
                    continue
                n_max = int(sub['n_temp_c'].max()) if 'n_temp_c' in sub.columns else 0
                ax.plot(sub['hour'], sub['mean_temp_c'], linewidth=1.5, color=color, 
                        label=f'{status} (n≈{n_max:,})')
                if 'se_temp_c' in sub.columns:
                    ax.fill_between(sub['hour'],
                                    sub['mean_temp_c'] - 1.96 * sub['se_temp_c'],
                                    sub['mean_temp_c'] + 1.96 * sub['se_temp_c'],
                                    color=color, alpha=0.15)
        traj_mort = mort_rates[idx]
        ax.set_xlabel('Hours')
        ax.set_ylabel('Temp (°C)')
        ax.set_title(f'D{idx+1}. {traj} (mort={traj_mort:.0f}%)',
                     fontsize=10, fontweight='bold', color=TRAJ_COLORS[traj])
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)
    
    fig.suptitle(f'OHCA Temperature Trajectories and Mortality — {n_sites} CLIF Sites ({window}h)',
                 fontsize=15, fontweight='bold', y=1.02)
    fig.savefig(OUTPUT_DIR / f'grant_summary_figure_{window}h.png', dpi=200, bbox_inches='tight')
    print(f'[OK] grant_summary_figure_{window}h.png')
    plt.show()

In [ ]:
# =============================================================
# FIGURE 8b: MORTALITY BY TRAJECTORY — ALL SITES + POOLED (DOT PLOT)
# =============================================================
for window in WINDOWS:
    t1 = data[f't1_{window}']
    if len(t1) == 0:
        continue
    
    sites = sorted(t1['site'].unique())
    
    fig, ax = plt.subplots(figsize=(12, 7))
    
    y_spacing = 1.0
    traj_gap = 2.0
    y_pos = {}
    y = 0
    
    for traj in TRAJ_ORDER:
        # Sites
        site_morts = []
        for site in sites:
            sd = t1[(t1['site'] == site) & (t1['group'] == traj)]
            m = sd[sd['variable'] == 'mortality_pct']['value']
            n = sd[sd['variable'] == 'n']['value']
            mort = float(m.iloc[0]) if len(m) > 0 else np.nan
            nn = int(n.iloc[0]) if len(n) > 0 else 0
            site_morts.append((site, mort, nn))
        
        # Sort sites by mortality within trajectory
        site_morts.sort(key=lambda x: x[1] if not np.isnan(x[1]) else 0)
        
        for site, mort, nn in site_morts:
            size = max(20, min(100, nn / 10))  # size by N
            ax.scatter(mort, y, s=size, color=TRAJ_COLORS[traj], alpha=0.6, 
                       edgecolors='white', zorder=3)
            ax.text(mort + 1.5, y, f'{site} ({nn})', va='center', fontsize=6, alpha=0.7)
            y += y_spacing
        
        # Pooled diamond
        pt = pooled_t1[window]
        pm = pt[(pt['group'] == traj) & (pt['variable'] == 'mortality_pct')]['value']
        pn = pt[(pt['group'] == traj) & (pt['variable'] == 'n')]['value']
        if len(pm) > 0:
            pooled_mort = float(pm.iloc[0])
            pooled_n = int(pn.iloc[0])
            ax.scatter(pooled_mort, y, s=150, color=TRAJ_COLORS[traj], marker='D', 
                       edgecolors='black', linewidth=1.5, zorder=5)
            ax.text(pooled_mort + 1.5, y, f'POOLED ({pooled_n:,})', va='center', 
                    fontsize=7, fontweight='bold')
        
        # Trajectory label
        mid_y = y - (len(site_morts) * y_spacing) / 2
        ax.text(-8, mid_y, traj, va='center', ha='right', fontsize=10, fontweight='bold',
                color=TRAJ_COLORS[traj])
        
        y += traj_gap  # gap between trajectory groups
        
        # Horizontal separator
        ax.axhline(y - traj_gap/2, color='gray', linewidth=0.3, alpha=0.5)
    
    ax.set_xlabel('Mortality (%)', fontsize=12)
    ax.set_xlim(-5, 105)
    ax.set_yticks([])
    ax.set_title(f'Site-Level Mortality by Trajectory ({window}h, {len(sites)} CLIF Sites)\n'
                 f'Circle size ∝ sample size | ◆ = Pooled estimate',
                 fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.2, axis='x')
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'dotplot_mortality_by_trajectory_{window}h.png', dpi=200, bbox_inches='tight')
    print(f'[OK] dotplot_mortality_by_trajectory_{window}h.png')
    plt.show()

In [ ]:
# Debug: find the spike
hv = data['hv_24']
check = hv[(hv['trajectory'] == 'Rapid Decline') & (hv['hour'].between(9, 11))]
print(check[['site', 'hour', 'survival_status', 'mean_temp_c', 'n_temp_c']].sort_values(['hour', 'site']).to_string())

In [ ]:
# Add before plotting:
sub = hv[(hv['site'] == site) & (hv['trajectory'] == traj)].copy()
sub = sub.dropna(subset=['mean_temp_c'])
sub = sub[(sub['mean_temp_c'] >= 32) & (sub['mean_temp_c'] <= 40)]  # clip implausible means
sub = sub[sub['n_temp_c'] >= 5]  # require minimum N

In [ ]:
# In the plotting loop, after selecting site + trajectory data:
sub = hv[(hv['site'] == site) & (hv['trajectory'] == traj)].copy()
sub = sub.dropna(subset=['mean_temp_c', 'n_temp_c'])
sub = sub[sub['n_temp_c'] >= 10]                           # minimum 10 patients
sub = sub[(sub['mean_temp_c'] >= 33) & (sub['mean_temp_c'] <= 40)]  # physiological range

In [ ]:
# =============================================================
# SUMMARY OF ALL FIGURES
# =============================================================
print('=' * 60)
print('  ALL GRANT FIGURES COMPLETE')
print('=' * 60)
print(f'\n  Output: {OUTPUT_DIR}')
print(f'\n  Figures generated:')
for f in sorted(OUTPUT_DIR.glob('*.png')):
    print(f'    {f.name:50s} {f.stat().st_size / 1024:>8.1f} KB')
print(f'\n  Recommended for grant:')
print(f'    1. grant_summary_figure_72h.png  — composite overview')
print(f'    2. forest_plot_mortality_72h.png  — site-level evidence')
print(f'    3. heatmap_mortality_72h.png      — visual heterogeneity')
print(f'    4. site_overview_72h.png          — consortium scope')